# Data Preparation Lab -- Module 2, Class 2

**Dataset:** Superstore Sales

In this lab you will:
1. Load and inspect data
2. Handle missing values
3. Remove duplicates
4. Convert data types
5. Create derived features

The first 3 tasks are pre-built. The rest are TODO for you.

In [16]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vivek468/superstore-dataset-final")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'superstore-dataset-final' dataset.
Path to dataset files: /kaggle/input/superstore-dataset-final


---
## Setup: Load the Dataset

Option A: Upload from Kaggle (download from https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

Option B: Use the URL loader below.

In [17]:
import pandas as pd
import numpy as np

# Option A: Upload in Colab
# from google.colab import files
# uploaded = files.upload()  # upload your CSV
# df = pd.read_csv('SampleSuperstore.csv')

# Option B: Load from a public URL
# If the URL does not work, use Option A.
url = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"

try:
    df = pd.read_csv(url, encoding='latin-1')
    print(f"Loaded from URL: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"URL failed ({e}). Use Option A: upload the CSV manually.")
    # Fallback: upload manually
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded from upload: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded from URL: 10800 rows, 21 columns


---
## Task 1: Inspect the Data (pre-built)

Always look at your data before doing anything to it.

In [ ]:
# First 5 rows
df.head(100)

In [20]:
import pandas as pd

# Define df by reloading the data to make this cell executable independently
url = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"
df = pd.read_csv(url, encoding='latin-1')

# Original code
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Shape: 10800 rows, 21 columns


In [21]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10800 non-null  object 
 1   Order ID       10800 non-null  object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9983 non-null   float64
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quanti

In [22]:
# Summary statistics for numerical columns
df.describe()

,Postal Code,Sales,Quantity,Discount,Profit
count,9983.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,55245.233297,229.858001,3.789574,0.156203,28.656896
std,32038.715955,623.245101,2.225110,0.206452,234.260108
min,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,57103.000000,54.490000,3.000000,0.200000,8.666500
75%,90008.000000,209.940000,5.000000,0.200000,29.364000
max,99301.000000,22638.480000,14.000000,0.800000,8399.976000


---
## Task 2: Missing Values (pre-built)

Check which columns have missing values and how many.

In [23]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

# Show only columns with missing values
missing_report[missing_report['missing_count'] > 0]

# Fill text columns
text_cols = ['Order Date','Ship Date','Ship Mode','Customer ID','Customer Name',
             'Segment','Country','City','State','Region','Product ID',
             'Category','Sub-Category','Product Name']

df[text_cols] = df[text_cols].fillna("Unknown")

# Fill numeric columns
num_cols = ['Sales','Quantity','Discount','Profit']
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

In [24]:
import numpy as np

# Fill missing numerical values with median (robust to outliers)
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val) # Changed to direct assignment
        print(f"Filled {col} missing values with median: {median_val}")

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val) # Changed to direct assignment
        print(f"Filled {col} missing values with mode: {mode_val}")

# Verify: no more missing values
print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")

Filled Postal Code missing values with median: 57103.0

Total missing values remaining: 0


---
## Task 3: Remove Duplicates (pre-built)

In [25]:
# Check for duplicates
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates}")

# Remove duplicates
if n_duplicates > 0:
    df = df.drop_duplicates()
    print(f"After removal: {df.shape[0]} rows remain")
else:
    print("No duplicates to remove.")

Duplicate rows found: 504
After removal: 10296 rows remain


---
## Task 4: Convert Date Columns (TODO)

The `Order Date` and `Ship Date` columns are stored as strings. Convert them to proper datetime objects.

Hint: Use `pd.to_datetime()`. After conversion, verify with `.dtypes`.

In [26]:
# Check current types of date columns
print("Before conversion:")
for col in df.columns:
    if 'date' in col.lower() or 'Date' in col:
        print(f"  {col}: {df[col].dtype}")
        print(f"  Sample value: {df[col].iloc[0]}")

Before conversion:
  Order Date: object
  Sample value: 11/8/2017
  Ship Date: object
  Sample value: 11/11/2017


In [27]:
# TODO: Convert date columns to datetime
# Look at the column names printed above and convert them.
# The column names may vary depending on the dataset version.
#
# Example pattern:
# df['Column Name'] = pd.to_datetime(df['Column Name'])

# Your code here:
for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

In [28]:
# TODO: Verify the conversion worked
# Print dtypes for the date columns to confirm they are datetime64

# Your code here:
print("After conversion:")
for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        print(f"  {col}: {df[col].dtype}")

After conversion:
  Order Date: datetime64[ns]
  Ship Date: datetime64[ns]


---
## Task 5: Derived Features (TODO)

Create customer-level summary features. These are the building blocks for customer segmentation (Activity 4).

You need to create:
- **total_spending**: Total sales per customer
- **order_frequency**: Number of orders per customer
- **avg_order_value**: Average sales amount per order per customer

Hint: Use `df.groupby('Customer ID')` (or whatever the customer ID column is named).

In [29]:
# First, identify the right column names
print("All columns:")
for col in df.columns:
    print(f"  {col}")

All columns:
  Row ID
  Order ID
  Order Date
  Ship Date
  Ship Mode
  Customer ID
  Customer Name
  Segment
  Country
  City
  State
  Postal Code
  Region
  Product ID
  Category
  Sub-Category
  Product Name
  Sales
  Quantity
  Discount
  Profit


In [30]:
# TODO: Create total_spending per customer
# Hint: df.groupby('Customer ID')['Sales'].sum()
#
# Replace column names below with the actual names from your dataset.

# customer_spending = df.groupby('???')['???'].sum()
# customer_spending.name = 'total_spending'

# Your code here:

customer_spending = df.groupby('Customer ID')['Sales'].sum()
customer_spending.name = 'total_spending'

print(customer_spending)

Customer ID
AA-10315    5563.560
AA-10375    1056.390
AA-10480    1790.512
AA-10645    5086.935
AB-10015     886.156
              ...   
XP-21865    2374.658
YC-21895    5454.350
YS-21880    6720.444
ZC-21910    8025.707
ZD-21925    1493.944
Name: total_spending, Length: 794, dtype: float64


In [31]:
# TODO: Create order_frequency per customer
# Hint: Count the number of rows (orders) per customer.
# Use .groupby(...).size() or .groupby(...)['some_col'].count()

# Your code here:
order_frequency = df.groupby('Customer ID').size()
order_frequency.name = 'order_frequency'

print(order_frequency)


Customer ID
AA-10315    11
AA-10375    15
AA-10480    12
AA-10645    18
AB-10015     6
            ..
XP-21865    28
YC-21895     8
YS-21880    12
ZC-21910    31
ZD-21925     9
Name: order_frequency, Length: 794, dtype: int64


In [32]:
# TODO: Create avg_order_value per customer
# Hint: Use .groupby(...)['Sales'].mean()

# Your code here:
avg_order_value = df.groupby('Customer ID')['Sales'].mean()
avg_order_value.name = 'avg_order_value'

print(avg_order_value)

Customer ID
AA-10315    505.778182
AA-10375     70.426000
AA-10480    149.209333
AA-10645    282.607500
AB-10015    147.692667
               ...    
XP-21865     84.809214
YC-21895    681.793750
YS-21880    560.037000
ZC-21910    258.893774
ZD-21925    165.993778
Name: avg_order_value, Length: 794, dtype: float64


In [33]:
# TODO: Combine all three into a single customer-level DataFrame
# Hint: Use pd.concat([series1, series2, series3], axis=1)
# or create them all at once with .groupby(...).agg(...)

# customer_summary = pd.concat([customer_spending, order_freq, avg_order], axis=1)
# customer_summary.columns = ['total_spending', 'order_frequency', 'avg_order_value']

# Your code here:

customer_summary = pd.concat(
    [customer_spending, order_frequency, avg_order_value],
    axis=1
)

customer_summary.columns = [
    'total_spending',
    'order_frequency',
    'avg_order_value'
]

print(customer_summary)


             total_spending  order_frequency  avg_order_value
Customer ID                                                  
AA-10315           5563.560               11       505.778182
AA-10375           1056.390               15        70.426000
AA-10480           1790.512               12       149.209333
AA-10645           5086.935               18       282.607500
AB-10015            886.156                6       147.692667
...                     ...              ...              ...
XP-21865           2374.658               28        84.809214
YC-21895           5454.350                8       681.793750
YS-21880           6720.444               12       560.037000
ZC-21910           8025.707               31       258.893774
ZD-21925           1493.944                9       165.993778

[794 rows x 3 columns]


In [34]:
# TODO: Display the first 10 rows of your customer summary and .describe()

# Your code here:

# First 10 rows
print(customer_summary.head(10))

# Summary statistics
print(customer_summary.describe())


             total_spending  order_frequency  avg_order_value
Customer ID                                                  
AA-10315           5563.560               11       505.778182
AA-10375           1056.390               15        70.426000
AA-10480           1790.512               12       149.209333
AA-10645           5086.935               18       282.607500
AB-10015            886.156                6       147.692667
AB-10060           7755.620               18       430.867778
AB-10105          14473.571               20       723.678550
AB-10150            966.710               12        80.559167
AB-10165           1113.838               14        79.559857
AB-10255            914.532               14        65.323714
       total_spending  order_frequency  avg_order_value
count      794.000000       794.000000       794.000000
mean      2980.627174        12.967254       227.870671
std       3531.879504        12.016652       190.222521
min          4.833000         1.

---
## Task 6: Save Cleaned Data (TODO)

Save the cleaned DataFrame to a new CSV file. Never overwrite the original.

In [35]:
# TODO: Save the cleaned main DataFrame
# df.to_csv('superstore_cleaned.csv', index=False)

# TODO: Save the customer summary DataFrame
# customer_summary.to_csv('customer_summary.csv')

# Your code here:

# Save cleaned main dataset
df.to_csv('superstore_cleaned.csv', index=False)

# Save customer summary dataset
customer_summary.to_csv('customer_summary.csv')

print("Files saved successfully!")


Files saved successfully!


---
## Reflection

Answer these in a text cell or comments:

1. Why did we use median instead of mean for filling numerical missing values?
2. What is the difference between the row-level DataFrame (one row per order) and the customer-level summary? When would you use each?
3. If two rows have identical values in every column, are they always true duplicates? When might they not be?
1. Why use median instead of mean for missing values?

We use the median because it is not affected by extreme values (outliers).

The mean (average) can be distorted if there are very high or very low values.
The median is the middle value, so it is more stable.

👉 So, median gives a more realistic and balanced replacement for missing data.

2. Difference between row-level DataFrame and customer-level summary
Row-level DataFrame (one row per order)
Each row represents one order
Contains detailed transaction data (order date, product, sales, etc.)

👉 Used for:

sales analysis
product performance
time trends (daily/weekly/monthly analysis)
Customer-level summary
Each row represents one customer
Combines all orders into customer metrics (total spending, frequency, average order value)

👉 Used for:

customer segmentation
marketing analysis
identifying high-value customers
3. Are identical rows always true duplicates?

Not always.

Two rows can look identical but still not be true duplicates when:

they represent different events recorded at the same time
data was copied from different systems
hidden identifiers (like transaction IDs) are different but not shown
time-sensitive systems log repeated identical actions (e.g., duplicate clicks or retries)

👉 So, identical values do not always mean the data is truly duplicated in meaning.

✅ Brief Summary
Median is used because it is more stable than mean.
Row-level data shows individual orders; customer-level shows customer behavior.
Identical rows may not always be real duplicates depending on context.